# Phân tích vận hành TikTok Shop – Histolab Việt Nam

Notebook chạy trên Google Colab để phân tích file Shop Analytics theo tháng, tạo KPI, biểu đồ và CSV phục vụ báo cáo thực tập. Tải file Excel khi được yêu cầu. Không dùng kỳ so sánh cũ trong file làm nghiên cứu trước–sau.

In [ ]:
!pip -q install openpyxl
import pandas as pd, numpy as np, matplotlib.pyplot as plt, re
from google.colab import files
pd.set_option('display.max_columns', 100)
uploaded = files.upload()
FILE_PATH = next(iter(uploaded.keys()))
print("Đang đọc:", FILE_PATH)


In [ ]:
# Đọc sheet đầu tiên và tìm dòng tiêu đề
raw = pd.read_excel(FILE_PATH, sheet_name=0, header=None)
# Chọn dòng tiêu đề có cột ngày và GMV; tránh nhầm bảng tổng quan ở đầu file
header_row = None
for i in range(min(len(raw), 30)):
    vals = [str(v).strip().lower() for v in raw.iloc[i].tolist()]
    has_date = any(v in ['ngày','date','day'] for v in vals)
    has_gmv = any(v == 'gmv' for v in vals)
    if has_date and has_gmv:
        header_row = i
        break
if header_row is None:
    keywords = ['gmv','đơn hàng','khách hàng','số món bán ra','hoàn tiền']
    scores = raw.astype(str).apply(lambda r: sum(any(k in str(v).lower() for v in r) for k in keywords), axis=1)
    header_row = int(scores.idxmax())
print("Dòng tiêu đề:", header_row, "| điểm:", int(scores.max()))
headers = raw.iloc[header_row].astype(str).str.strip().tolist()
df = raw.iloc[header_row+1:].copy()
df.columns = headers
df = df.loc[:, ~pd.Index(df.columns).duplicated()].dropna(how='all').reset_index(drop=True)
display(df.head())
print(df.columns.tolist())


In [ ]:
# Chuẩn hóa ngày và các cột số
def norm(x): return re.sub(r'\\s+', ' ', str(x).strip()).lower()
date_col = next((c for c in df.columns if norm(c) in ['date','day','time'] or 'date' in norm(c)), df.columns[0])
df = df.rename(columns={date_col:'Date'})
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)
df = df[df['Date'].notna()].copy()
for c in df.columns:
    if c != 'Date' and df[c].dtype == 'object':
        x = df[c].astype(str).str.replace(',', '', regex=False).str.replace('%','',regex=False).str.strip()
        n = pd.to_numeric(x, errors='coerce')
        if n.notna().sum(): df[c] = n
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df = df.sort_values('Date').reset_index(drop=True)
print("Khoảng ngày:", df.Date.min(), "→", df.Date.max(), "| số dòng:", len(df))


In [ ]:
# Ánh xạ các trường KPI theo tên cột
def find_col(*terms):
    # Hỗ trợ cả tiêu đề tiếng Anh và tiếng Việt trong file Shop Analytics
    aliases = {
      'gmv':['gmv'], 'orders':['orders','đơn hàng'], 'customers':['customers','khách hàng'],
      'items sold':['items sold','số món bán ra'], 'refunds':['refunds','hoàn tiền'],
      'total revenue':['total revenue','tổng doanh thu'], 'page views':['page views','lượt xem trang'],
      'visitors':['visitors','khách truy cập'], 'conversion rate':['conversion rate','tỷ lệ chuyển đổi'],
      'product impressions':['product impressions','lượt hiển thị sản phẩm'],
      'product clicks':['product clicks','lượt nhấp vào sản phẩm'], 'aov':['aov'],
      'linked video gmv':['linked video gmv','gmv đến từ video liên kết'],
      'creator video gmv':['creator video gmv','gmv video của nhà sáng tạo'],
      'creator live gmv':['creator live gmv','gmv live của nhà sáng tạo']
    }
    wanted = [t.lower() for t in terms]
    for c in df.columns:
        name = norm(c)
        if all(any(a in name for a in aliases.get(t, [t])) for t in wanted): return c
    return None
C = {
 'GMV':find_col('gmv'), 'Orders':find_col('orders'), 'Customers':find_col('customers'),
 'Items sold':find_col('items sold'), 'Refunds':find_col('refunds'),
 'Total revenue':find_col('total revenue'), 'Page views':find_col('page views'),
 'Visitors':find_col('visitors'), 'Conversion rate':find_col('conversion rate'),
 'Product impressions':find_col('product impressions'), 'Product clicks':find_col('product clicks'),
 'AOV':find_col('aov'), 'Linked video GMV':find_col('linked video gmv'),
 'Creator video GMV':find_col('creator video gmv'), 'Creator LIVE GMV':find_col('creator live gmv')
}
display(pd.DataFrame({'KPI':list(C.keys()),'Cột được nhận diện':list(C.values())}))
for kpi_col in C.values():
    if kpi_col is not None:
        df[kpi_col] = pd.to_numeric(df[kpi_col].astype(str).str.replace(',', '', regex=False).str.replace('%', '', regex=False).str.strip(), errors='coerce')


In [ ]:
# Tổng hợp theo tháng: cộng các chỉ tiêu dạng lượng/giá trị; lấy trung bình cho CVR và AOV
sum_keys = ['GMV','Orders','Customers','Items sold','Refunds','Total revenue','Page views','Visitors',
            'Product impressions','Product clicks','Linked video GMV','Creator video GMV','Creator LIVE GMV']
agg = {k:(C[k],'sum') for k in sum_keys if C[k] is not None}
monthly = df.groupby('Month',as_index=False).agg(**agg)
for k in ['Conversion rate','AOV']:
    if C[k] is not None:
        temp = df.groupby('Month')[C[k]].mean().rename(k+' (daily mean)').reset_index()
        monthly = monthly.merge(temp,on='Month',how='left')
if C['Product impressions'] and C['Product clicks']:
    monthly['CTR recalculated (%)'] = monthly['Product clicks']/monthly['Product impressions']*100
if C['Refunds'] and C['Total revenue']:
    monthly['Refund value / revenue (%)'] = monthly['Refunds']/monthly['Total revenue']*100
display(monthly)


In [ ]:
# KPI tổng hợp toàn kỳ
rows=[]
for k in sum_keys:
    if C[k] is not None:
        s=pd.to_numeric(df[C[k]],errors='coerce')
        rows.append({'Chỉ tiêu':k,'Tổng toàn kỳ':s.sum(),'Trung bình mỗi dòng ngày':s.mean()})
if C['Product impressions'] and C['Product clicks']:
    rows.append({'Chỉ tiêu':'CTR toàn kỳ (%)','Tổng toàn kỳ':df[C['Product clicks']].sum()/df[C['Product impressions']].sum()*100,'Trung bình mỗi dòng ngày':np.nan})
if C['Refunds'] and C['Total revenue']:
    rows.append({'Chỉ tiêu':'Tỷ lệ giá trị hoàn tiền (%)','Tổng toàn kỳ':df[C['Refunds']].sum()/df[C['Total revenue']].sum()*100,'Trung bình mỗi dòng ngày':np.nan})
summary=pd.DataFrame(rows)
display(summary)


In [ ]:
# Biểu đồ 1: GMV theo tháng
if 'GMV' in monthly:
    plt.figure(figsize=(12,4)); plt.plot(monthly['Month'],monthly['GMV'],marker='o')
    plt.title('GMV theo tháng'); plt.xlabel('Tháng'); plt.ylabel('GMV (VND)')
    plt.xticks(rotation=45); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


In [ ]:
# Biểu đồ 2: đơn hàng, khách hàng, sản phẩm bán ra
metrics=[k for k in ['Orders','Customers','Items sold'] if k in monthly]
if metrics:
    monthly.set_index('Month')[metrics].plot(kind='bar',figsize=(12,4))
    plt.title('Đơn hàng, khách hàng và sản phẩm bán ra'); plt.xlabel('Tháng')
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()


In [ ]:
# Biểu đồ 3: traffic và tương tác sản phẩm
metrics=[k for k in ['Visitors','Page views','Product impressions','Product clicks'] if k in monthly]
if metrics:
    monthly.set_index('Month')[metrics].plot(figsize=(12,4),marker='o')
    plt.title('Traffic và tương tác sản phẩm theo tháng'); plt.xlabel('Tháng')
    plt.xticks(rotation=45); plt.grid(alpha=.25); plt.tight_layout(); plt.show()
if 'Conversion rate (daily mean)' in monthly:
    monthly.plot(x='Month',y='Conversion rate (daily mean)',marker='o',figsize=(12,4),legend=False)
    plt.title('CVR theo tháng (trung bình các dòng ngày)'); plt.xticks(rotation=45); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


In [ ]:
# Biểu đồ 4: hoàn tiền
if 'Refunds' in monthly:
    monthly.plot(x='Month',y='Refunds',kind='bar',figsize=(12,4),legend=False)
    plt.title('Giá trị hoàn tiền theo tháng'); plt.ylabel('Giá trị'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print('Lưu ý: dữ liệu tổng hợp chưa đủ để tính tỷ lệ đơn hủy/trả hàng hoặc thời gian CSKH nếu thiếu dữ liệu chi tiết từng đơn.')


In [ ]:
# Biểu đồ 5: các chỉ tiêu GMV nội dung – không cộng vì có thể khác quy tắc quy thuộc
metrics=[k for k in ['Linked video GMV','Creator video GMV','Creator LIVE GMV'] if k in monthly]
if metrics:
    monthly.set_index('Month')[metrics].plot(figsize=(12,4),marker='o')
    plt.title('Các chỉ tiêu GMV theo nguồn nội dung'); plt.xlabel('Tháng')
    plt.xticks(rotation=45); plt.grid(alpha=.25); plt.tight_layout(); plt.show()
print('Không cộng các trường GMV nội dung khi chưa xác minh định nghĩa và khả năng trùng quy thuộc.')


In [ ]:
# Xuất file phục vụ báo cáo và dashboard
df.to_csv('cleaned_daily_data.csv',index=False,encoding='utf-8-sig')
monthly.to_csv('monthly_kpi.csv',index=False,encoding='utf-8-sig')
summary.to_csv('summary_kpi.csv',index=False,encoding='utf-8-sig')
files.download('cleaned_daily_data.csv')
files.download('monthly_kpi.csv')
files.download('summary_kpi.csv')


## Lưu ý phương pháp
- Phân tích phạm vi ngày có trong file, dự kiến 01/09/2025–30/06/2026.
- Không diễn giải biến động KPI thành quan hệ nhân quả nếu chưa đối chiếu lịch chiến dịch và dữ liệu liên quan.
- CVR/AOV là tỷ lệ/giá trị trung bình; không cộng theo tháng.
- Tỷ lệ hoàn tiền theo giá trị không đồng nghĩa tỷ lệ đơn hàng được hoàn tiền.
- Kiểm tra dòng tiêu đề và tên cột nếu cấu trúc file Excel thay đổi.

## Bổ sung: KPI mục tiêu cho tuần đầu tháng 7, tháng 7 và Quý III/2026
Giữ nguyên các phân tích và biểu đồ đã có; cell dưới chỉ bổ sung bảng KPI mục tiêu tăng 10%.

In [ ]:
# Bổ sung: KPI mục tiêu tăng 10% cho tuần đầu tháng 7, tháng 7 và Quý III/2026
# Công thức: KPI mục tiêu = giá trị kỳ cơ sở × 1.10
# Mốc dữ liệu lịch sử kết thúc 30/06/2026.
# Cell này nhận diện cột ngày và KPI từ df đã được tạo ở các cell phía trên.
import re
import numpy as np
import pandas as pd

def _n(x):
    return re.sub(r"\s+", " ", str(x).strip()).lower()

date_col = next((c for c in df.columns if _n(c) in ["date", "day", "time", "ngày"] or "date" in _n(c) or "ngày" in _n(c)), None)
if date_col is None and "Date" in df.columns:
    date_col = "Date"
if date_col is None:
    raise ValueError("Không tìm thấy cột ngày trong df. Hãy kiểm tra các cell chuẩn hóa dữ liệu trước đó.")

kpi_candidates = {
    "GMV": ["gmv"],
    "Orders": ["orders", "đơn hàng"],
    "Customers": ["customers", "khách hàng"],
    "Items sold": ["items sold", "số món bán ra", "sản phẩm bán"]
}
found = {}
for k, candidates in kpi_candidates.items():
    found[k] = next((c for c in df.columns if any(term in _n(c) for term in candidates)), None)

hist = df.copy()
hist["_Date"] = pd.to_datetime(hist[date_col], errors="coerce", dayfirst=True)
hist = hist[hist["_Date"].notna()].copy()
for k, c in found.items():
    if c:
        hist[c] = pd.to_numeric(hist[c].astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False), errors="coerce")

last_day = hist["_Date"].max().normalize()
if last_day.month != 6:
    print("Cảnh báo: dữ liệu mới nhất không kết thúc tháng 6; code vẫn lấy ngày cuối cùng trong file làm mốc.")
june = hist[(hist["_Date"].dt.year == last_day.year) & (hist["_Date"].dt.month == last_day.month)].copy()
last3_start = (last_day.to_period("M") - 2).start_time
last3 = hist[(hist["_Date"] >= last3_start) & (hist["_Date"] <= last_day)].copy()

def _sum(data, col):
    return pd.to_numeric(data[col], errors="coerce").sum() if col and not data.empty else np.nan

def _daily_mean(data, col):
    if not col or data.empty:
        return np.nan
    daily = data.groupby(data["_Date"].dt.date)[col].sum(min_count=1)
    return daily.mean()

forecast_rows = []
# Tuần đầu tháng 7 được xác định là 01–05/07/2026 (5 ngày).
for metric, col in found.items():
    if not col:
        continue
    base = _daily_mean(june, col) * 5
    forecast_rows.append({
        "Tần suất": "Tuần đầu tháng 7 (01–05/07/2026)",
        "Chỉ tiêu": metric,
        "Kỳ cơ sở": f"Bình quân ngày tháng {last_day:%m/%Y} × 5 ngày",
        "Giá trị cơ sở": base,
        "Tỷ lệ tăng (%)": 10,
        "KPI mục tiêu": base * 1.10 if pd.notna(base) else np.nan,
        "Công thức": "Bình quân ngày tháng 6 × 5 × 1,10"
    })
    base_month = _sum(june, col)
    forecast_rows.append({
        "Tần suất": "Tháng 7/2026",
        "Chỉ tiêu": metric,
        "Kỳ cơ sở": f"Thực tế tháng {last_day:%m/%Y}",
        "Giá trị cơ sở": base_month,
        "Tỷ lệ tăng (%)": 10,
        "KPI mục tiêu": base_month * 1.10 if pd.notna(base_month) else np.nan,
        "Công thức": "Thực tế tháng 6 × 1,10"
    })
    base_q = _sum(last3, col)
    forecast_rows.append({
        "Tần suất": "Quý III/2026 (07–09/2026)",
        "Chỉ tiêu": metric,
        "Kỳ cơ sở": "Tổng thực tế 3 tháng gần nhất trong file",
        "Giá trị cơ sở": base_q,
        "Tỷ lệ tăng (%)": 10,
        "KPI mục tiêu": base_q * 1.10 if pd.notna(base_q) else np.nan,
        "Công thức": "Tổng thực tế 3 tháng 4–6 × 1,10"
    })

kpi_targets_next = pd.DataFrame(forecast_rows)
display(kpi_targets_next)
kpi_targets_next.to_csv("kpi_targets_july_q3_2026_plus_10pct.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("kpi_targets_july_q3_2026_plus_10pct.csv")
